# Part 5 · Handling bad input at inference time

A wrong answer that looks right is worse than an error. This notebook shows one, then the checks that stop it.

In [1]:
from pathlib import Path

import churn_model

if not Path("churn_model.joblib").exists():
    churn_model.main()  # trains in THIS environment and saves the model
else:
    print("churn_model.joblib already exists")

churn_model.joblib already exists


## A quiet failure
The encoder ignores categories it has not seen, so a contract typed with a capital letter still scores.

In [2]:
import pandas as pd

from predict import model

row = pd.DataFrame(
    [
        {
            "tenure_months": 3,
            "monthly_charge": 95.0,
            "support_calls": 4,
            "contract": "month-to-month",
            "payment": "cheque",
        }
    ]
)
print("spelled correctly:", model.predict_proba(row).round(3))
row["contract"] = "Month-to-month"
print("capital letter   :", model.predict_proba(row).round(3))

spelled correctly: [[0.071 0.929]]
capital letter   : [[0.134 0.866]]


## The checks in predict.py
`predict_churn` calls `check_inputs` first, which raises a `ValueError` with a message for the person who typed the input.

In [3]:
from predict import predict_churn

cases = [
    (-5, 95.0, 4, "month-to-month", "cheque"),
    (None, 95.0, 4, "month-to-month", "cheque"),
    (600, 95.0, 4, "month-to-month", "cheque"),
    (3, 95.0, 4, "weekly", "cheque"),
    (3, 95.0, 4, "Month-to-month", "cheque"),
    (3, None, 4, "month-to-month", "cheque"),
]
for case in cases:
    try:
        print(case, "->", round(predict_churn(*case), 3))
    except ValueError as error:
        print(case, "-> ValueError:", error)

(-5, 95.0, 4, 'month-to-month', 'cheque') -> ValueError: Tenure must be a number from 0 to 120 months.
(None, 95.0, 4, 'month-to-month', 'cheque') -> ValueError: Tenure must be a number from 0 to 120 months.
(600, 95.0, 4, 'month-to-month', 'cheque') -> ValueError: Tenure must be a number from 0 to 120 months.
(3, 95.0, 4, 'weekly', 'cheque') -> ValueError: Contract must be one of: month-to-month, one-year, two-year.
(3, 95.0, 4, 'Month-to-month', 'cheque') -> ValueError: Contract must be one of: month-to-month, one-year, two-year.
(3, None, 4, 'month-to-month', 'cheque') -> 0.898


A missing monthly charge is **accepted**, because the pipeline was trained to handle it. A missing tenure is **rejected**, because it was not.

In [4]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no"], capture_output=True, text=True
)
print(result.stdout)

.......                                                                  [100%]
7 passed in 1.53s



**Try it:** run `python app.py`, enter a tenure of -5 and read the pop-up. Then remove the `try` and `except` from `score` and compare what the page shows with what the terminal shows.